In [12]:
import numpy as np
import torch


A = np.random.randn(3, 4).astype(np.float32)
# B = np.random.randn(3, 4).astype(np.float32)
B = np.random.randn(4, 5).astype(np.float32)
C = np.random.randn(3, 5).astype(np.float32)

At = torch.tensor(A, requires_grad=True)
Bt = torch.tensor(B, requires_grad=True)
Ct = torch.tensor(C, requires_grad=True)

h = At @ Bt
out = h + Ct
# out = out.swapaxes(-2, -1)
loss = out.sum()
# out = At.sort().values
# print(At, out)
# grad = np.ones_like(out.detach().numpy())
# out.backward(gradient=torch.tensor(grad))
loss.backward(create_graph=False, retain_graph=False)

At.grad, Bt.grad, out, h

(tensor([[-2.1147,  0.6906, -0.8253, -4.5955],
         [-2.1147,  0.6906, -0.8253, -4.5955],
         [-2.1147,  0.6906, -0.8253, -4.5955]]),
 tensor([[ 0.6714,  0.6714,  0.6714,  0.6714,  0.6714],
         [ 0.2573,  0.2573,  0.2573,  0.2573,  0.2573],
         [-2.0064, -2.0064, -2.0064, -2.0064, -2.0064],
         [ 3.7758,  3.7758,  3.7758,  3.7758,  3.7758]]),
 tensor([[-0.6119,  1.8148, -0.8670, -3.2021,  0.1460],
         [ 0.6200, -2.0378, -0.3484, -2.1074, -0.6262],
         [ 0.7311, -4.9160, -2.5704, -1.8769, -0.2699]], grad_fn=<AddBackward0>),
 tensor([[ 0.6677, -0.2195, -1.6579, -2.9855,  0.4954],
         [ 1.0394, -2.0606, -2.0342, -1.7932, -0.5480],
         [ 1.3482, -4.6714, -2.6424, -2.3825,  0.5062]], grad_fn=<MmBackward0>))

In [8]:
import cattorch

Am = cattorch.Tensor(A, requires_grad=True)
Bm = cattorch.Tensor(B, requires_grad=True)
out :cattorch.Tensor = Am @ Bm
# out = out.swapaxes(-2, -1)
loss = out.sum()
cattorch.Tensor.build_graph_enabled()
# grad = cattorch.ones_like(out)
loss.backward(create_graph=False, retain_graph=False)
print(cattorch.Tensor.build_graph_enabled())

# a_grad = Am.grad
# a_grad.sum().backward()
Am.grad, Bm.grad, out

True


(Tensor(Array([[ 1.13157 -0.08552 -0.95502  2.20879]
        [ 1.13157 -0.08552 -0.95502  2.20879]
        [ 1.13157 -0.08552 -0.95502  2.20879]], dtype=float32)),
 Tensor(Array([[ 1.95945  1.95945  1.95945  1.95945  1.95945]
        [ 2.48491  2.48491  2.48491  2.48491  2.48491]
        [ 0.03021  0.03021  0.03021  0.03021  0.03021]
        [-2.17794 -2.17794 -2.17794 -2.17794 -2.17794]], dtype=float32)),
 Tensor(Array([[-2.46668 -2.07251  0.37987  1.46618 -0.48902]
        [-0.55026  0.02189 -0.00258  1.67127  0.14101]
        [-0.60745 -0.76064 -0.11044  0.38662  0.15803]], dtype=float32), requires_grad=True))

In [3]:
import cattorch

a = cattorch.Tensor([1.0, 0.0, -1e40], requires_grad=True)
b = cattorch.Tensor([2.0, 1.0, -2.0], requires_grad=True)

c = a/b
cattorch.Tensor.build_graph_enabled()
with cattorch.no_grad():
    c.backward(grad = cattorch.Tensor([2.,  1., -2.]), create_graph=False)
a.grad, b.grad, c.grad, cattorch.Tensor.build_graph_enabled()

/home/chang/.projs/cat-torch-from-scratch/cattorch/backend/_array.py:171: RuntimeWarning: overflow encountered in cast
  self._array = self._array.astype(dtype)


(Tensor(Array([1.00000 1.00000 1.00000], dtype=float32), requires_grad=True),
 Tensor(Array([-0.50000 -0.00000     -inf], dtype=float32), requires_grad=True),
 Tensor(Array([ 2.00000  1.00000 -2.00000], dtype=float32)),
 True)

In [17]:
import torch

t = torch.ones((10,), requires_grad=True)
v = torch.randint_like(t,2,10)
delta_s = v*t

x = delta_s.cumsum(dim=0)
s = x.sum()
s_ = torch.autograd.grad(s, t, create_graph=True)
s_, v, x

((tensor([90., 72., 48., 63., 54., 40., 12.,  6., 10.,  2.]),),
 tensor([9., 8., 6., 9., 9., 8., 3., 2., 5., 2.]),
 tensor([ 9., 17., 23., 32., 41., 49., 52., 54., 59., 61.],
        grad_fn=<CumsumBackward0>))